# 04 - DPO (Direct Preference Optimization) Eğitimi

**Amaç:** `Model_SFT` üzerinden başlayarak, `Anthropic/hh-rlhf` (Bai vd., 2022)
tercih verisiyle **doğrudan tercih optimizasyonu** (Rafailov vd., 2023)
uygulamak ve `Model_DPO`'yu elde etmek. DPO, ayrı bir ödül modeli veya
pekiştirmeli öğrenme (RL) döngüsü gerektirmeden, basit bir sınıflandırma
kaybıyla aynı hizalama hedefine ulaşmayı amaçlar; bu nedenle PPO'ya kıyasla
daha stabil ve hesaplama açısından daha hafiftir.

> Bu defter Google Colab Pro (A100/L4 GPU) üzerinde çalıştırılmak üzere tasarlanmıştır.

## 0. Ortam Kurulumu

In [ ]:
# --- Google Colab ortam kurulumu -------------------------------------------
# Bu hucre, projenin tamamini (Datasets/, src/, notebooks/, outputs/) iceren
# "Uygulama" klasorunu Google Drive'a yukledikten sonra calistirilmalidir.
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # TODO: Kendi Google Drive yolunuza gore guncelleyin
    PROJECT_ROOT = "/content/drive/MyDrive/Uygulama"
else:
    PROJECT_ROOT = os.path.abspath("..")

SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IN_COLAB:", IN_COLAB)

In [ ]:
# NOT: Surumler, TRL'nin klasik `PPOTrainer` + `AutoModelForCausalLMWithValueHead`
# coklu-adaptor (multi-adapter) RL ozelligiyle dogrulanmis, bilinen-uyumlu bir
# kombinasyon olacak sekilde SABITLENMISTIR (API kararliligi icin).
%pip install -q -U "transformers==4.44.2" "trl==0.9.6" "peft==0.11.1" "bitsandbytes==0.43.1" "accelerate==0.32.1" "datasets==2.20.0" "numpy==1.26.4" sentencepiece

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import alignment_pipeline as ap

ap.set_global_seed(42)
print("Cihaz:", ap.get_device())

In [ ]:
config = ap.ExperimentConfig(
    dataset_root=os.path.join(PROJECT_ROOT, "Datasets"),
    output_root=os.path.join(PROJECT_ROOT, "outputs"),
)
ap.ensure_dir(config.output_root)
print("Taban model:", config.base_model_id)
print("Veri kok dizini:", config.dataset_root)
print("Cikti kok dizini:", config.output_root)

In [ ]:
SFT_MERGED_DIR = config.path("sft", "merged_model")  # 02_SFT_Training.ipynb çıktısı (Model_SFT)
assert os.path.isdir(SFT_MERGED_DIR), "Önce 02_SFT_Training.ipynb çalıştırılmalı (Model_SFT bulunamadı)."

DPO_DIR = config.path("dpo")
DPO_ADAPTER_DIR = os.path.join(DPO_DIR, "adapter")
DPO_LOGS_DIR = os.path.join(DPO_DIR, "logs")
for d in (DPO_ADAPTER_DIR, DPO_LOGS_DIR):
    ap.ensure_dir(d)

QUANT_BITS = 4

## 1. Veri Hazırlığı

In [ ]:
hh = ap.load_hh_rlhf_preference(config.dataset_root, subsets=config.hh_rlhf_subsets)
dpo_dataset = ap.build_dpo_dataset(hh)
print({k: len(v) for k, v in dpo_dataset.items()})
print(dpo_dataset["train"][0])

## 2. Tokenizer ve LoRA Yapılandırması

In [ ]:
dpo_tokenizer = ap.load_tokenizer(SFT_MERGED_DIR)
dpo_lora_config = ap.make_lora_config(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=ap.QWEN2_LORA_TARGET_MODULES,
    task_type="CAUSAL_LM",
)

## 3. DPO Eğitimi

`beta` parametresi, politikanın referans (Model_SFT) politikasından ne kadar
uzaklaşabileceğini kontrol eder (yüksek beta → daha az sapma).

In [ ]:
trainer, dpo_model = ap.run_dpo(
    model_id=SFT_MERGED_DIR,
    train_dataset=dpo_dataset["train"],
    eval_dataset=dpo_dataset["test"],
    tokenizer=dpo_tokenizer,
    lora_config=dpo_lora_config,
    output_dir=DPO_DIR,
    beta=0.1,
    num_train_epochs=1.0,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_length=1024,
    max_prompt_length=config.max_prompt_length,
    quant_bits=QUANT_BITS,
)

## 4. Eğitim Eğrisi (Nicel)

In [ ]:
dpo_log_df = ap.trainer_log_history_to_df(trainer)
ap.save_json(trainer.state.log_history, os.path.join(DPO_LOGS_DIR, "dpo_log_history.json"))
ap.plot_loss_curve(dpo_log_df, loss_col="loss", title="DPO Eğitim Kaybı (Model_DPO)")

In [ ]:
dpo_eval_metrics = trainer.evaluate()
print(dpo_eval_metrics)
ap.save_json(dpo_eval_metrics, os.path.join(DPO_LOGS_DIR, "dpo_eval_metrics.json"))

## 5. Adaptörü Kaydet

In [ ]:
dpo_model.save_pretrained(DPO_ADAPTER_DIR)
dpo_tokenizer.save_pretrained(DPO_ADAPTER_DIR)
print("DPO adaptörü kaydedildi ->", DPO_ADAPTER_DIR)

## 6. Sonuç

`Model_DPO` = Model_SFT (taban) + `dpo_adapter` (LoRA). Artık dört model de
hazır: `Model_Base`, `Model_SFT`, `Model_RLHF`, `Model_DPO`. Sıradaki adım:
`05_Algorithm_Comparison_Quantitative.ipynb` ve
`06_Algorithm_Comparison_Qualitative.ipynb`.